In [8]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"

if not env_path.exists():
    raise FileNotFoundError(
        "Could not find .env. Open the notebook from the LangChain repository."
    )

load_dotenv(env_path)
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise RuntimeError(
        "GROQ_API_KEY is missing or empty in the repository .env file."
    )

## Completion model

In [9]:
from langchain_groq import ChatGroq

llmModel = ChatGroq(
    model="openai/gpt-oss-120b",
    groq_api_key=groq_api_key,
)

# chat completion model

In [10]:
chatModel = llmModel

## Prompts and Prompt Templates

In [11]:
# This is for completions model
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    "Tell me a {adjective} story about {topic}"
)

llmModelPrompt = prompt_template.format(
    adjective="curious",
    topic="Tesla"
)

res = llmModel.invoke(llmModelPrompt)
print(res)

content='**The Night the “White Star” Pigeons Saved a Genius**\n\n*It was a stormy October evening in 1895, and the air in New York City crackled not just with lightning but with the restless energy of an inventor who refused to sleep.*\n\nNikola Tesla had just finished a marathon of experiments in his modest Manhattan laboratory on West 37th Street. He was working on a new type of alternating‑current transformer that, if successful, would make it possible to transmit electricity across the Atlantic without the massive losses that plagued the era’s power lines. The prototype was a hulking, copper‑wound beast that hissed and sparked whenever he fed it a pulse of current.\n\nBut that night, something else caught his attention—a soft cooing from the fire escape outside his window. A flock of **white‑feathered pigeons**, their plumage almost luminous against the rain‑slicked brick, had taken refuge on the ledge. Among them was a particular bird that Tesla had named *“Mira.”* He had first r

In [12]:
# chat completion model

from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an {profession} expert on {topic}."),
        ("human", "Hello, Mr. {profession}, can you please answer a question?"),
        ("ai", "Sure!"),
        ("human", "{user_input}"),
    ]
)

messages = chat_template.format_messages(
    profession="Historian",
    topic="The Kennedy family",
    user_input="How many grandchildren had Joseph P. Kennedy?"
)

response = chatModel.invoke(messages)
print(response.content)


Joseph P. Kennedy Sr. (1888‑1969) and his wife Rose Fitzgerald Kennedy had nine children.  By the time all of their children had grown up, the Kennedy‑Kennedy line produced **22 grandchildren**.

| Kennedy child | Grandchildren (names) | Count |
|---------------|-----------------------|-------|
| **John F. Kennedy** (1917‑1963) | Caroline Kennedy (b. 1957), John F. Kennedy Jr. (1960‑1999) | 2 |
| **Eunice Kennedy Shriver** (1921‑2009) | Robert Kennedy Shriver (b. 1947), Maria Kennedy Shriver (b. 1954), Rosemary Kennedy Shriver (b. 1955), John Kennedy Shriver (b. 1956) | 4 |
| **Patricia Kennedy Lawford** (1924‑2006) | Christopher Lawford (b. 1955), Sydney Lawford (b. 1956) | 2 |
| **Robert F. Kennedy** (1925‑1968) | Kathleen Kennedy (b. 1951), Joseph Kennedy II (b. 1952), Robert F. Kennedy Jr. (b. 1954), David Kennedy (b. 1955), Michael Kennedy (b. 1958), Kerry Kennedy (b. 1960) | 6 |
| **Jean Kennedy Smith** (1928‑2020) | Stephen Smith (b. 1953), Christopher Smith (b. 1955) | 2 |
| **

# Old way

In [13]:
from langchain_core.messages import SystemMessage
from langchain_core.prompts import HumanMessagePromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(
            content=(
                "You are an Historian expert on the Kennedy family."
            )
        ),
        HumanMessagePromptTemplate.from_template("{user_input}"),
    ]
)

messages = chat_template.format_messages(
    user_input="Name the children and grandchildren of Joseph P. Kennedy?"
)

response = chatModel.invoke(messages)

In [14]:
response

AIMessage(content='**Joseph P. Kennedy\u202fSr. (1888‑1969)** – the patriarch of the Kennedy political dynasty – and his wife **Rose\u202fFitzgerald Kennedy (1890‑1995)** had nine children. Below are those children and, grouped by each child, the grandchildren they produced (as of the most widely‑recorded historical record).\n\n---\n\n## 1. Joseph\u202fKennedy\u202fJr. (1915‑1944) – “Joe”\n* **Married:** Rose Mary “Mimi” Hargrove (1938) – no children.  \n* **Note:** Died in a World‑War‑II aircraft training accident; therefore he has **no grandchildren**.\n\n---\n\n## 2. John\u202fF.\u202fKennedy (1917‑1963) – 35th President of the United States  \n* **Married:** Jacqueline Bouvier (1953)  \n* **Children (grandchildren of Joseph\u202fSr.):**  \n\n| Name | Birth‑Death | Notes |\n|------|-------------|-------|\n| **Caroline\u202fKennedy** | 1957‑present | Author, diplomat, former U.S. Ambassador to Australia. |\n| **John\u202fF.\u202fKennedy\u202fJr.** | 1960‑1999 | Died in a skiing accid

In [15]:
print(response.content)

**Joseph P. Kennedy Sr. (1888‑1969)** – the patriarch of the Kennedy political dynasty – and his wife **Rose Fitzgerald Kennedy (1890‑1995)** had nine children. Below are those children and, grouped by each child, the grandchildren they produced (as of the most widely‑recorded historical record).

---

## 1. Joseph Kennedy Jr. (1915‑1944) – “Joe”
* **Married:** Rose Mary “Mimi” Hargrove (1938) – no children.  
* **Note:** Died in a World‑War‑II aircraft training accident; therefore he has **no grandchildren**.

---

## 2. John F. Kennedy (1917‑1963) – 35th President of the United States  
* **Married:** Jacqueline Bouvier (1953)  
* **Children (grandchildren of Joseph Sr.):**  

| Name | Birth‑Death | Notes |
|------|-------------|-------|
| **Caroline Kennedy** | 1957‑present | Author, diplomat, former U.S. Ambassador to Australia. |
| **John F. Kennedy Jr.** | 1960‑1999 | Died in a skiing accident. |
| **Patrick Kennedy** | 1967‑present | Lawyer, former political candidate. |
| **Ara

# Few Shot Prompting

In [16]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "hi!", "output": "¡hola!"},
    {"input": "bye!", "output": "¡adiós!"},
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an English-Spanish translator."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)

# Chains

In [17]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "hi!", "output": "¡hola!"},
    {"input": "bye!", "output": "¡adiós!"},
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an English-Spanish translator."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)


chain = final_prompt | chatModel

res = chain.invoke({"input": "How are you?"})
print(res.content)

¿Cómo estás?


In [18]:
res

AIMessage(content='¿Cómo estás?', response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 117, 'total_tokens': 213, 'completion_time': 0.203861005, 'completion_tokens_details': {'reasoning_tokens': 83}, 'prompt_time': 0.005015468, 'prompt_tokens_details': None, 'queue_time': 0.198619732, 'total_time': 0.208876473}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_0708ac49a5', 'finish_reason': 'stop', 'logprobs': None}, id='run-5e5968c4-956a-4850-b134-6154bf4ca770-0', usage_metadata={'input_tokens': 117, 'output_tokens': 96, 'total_tokens': 213})

In [ ]:
# Langchain Expressions Language

# Output Parsers

In [19]:
from langchain_groq import ChatGroq

llmModel = ChatGroq(
    model="openai/gpt-oss-120b",
    groq_api_key=groq_api_key,
)

In [20]:
chatModel = llmModel

In [21]:
from langchain_core.prompts import PromptTemplate
from langchain.output_parsers.json import SimpleJsonOutputParser

json_prompt = PromptTemplate.from_template(
    "Return a JSON object with an `answer` key that answers the following question: {question}"
)

json_parser = SimpleJsonOutputParser()

json_chain = json_prompt | llmModel | json_parser

In [22]:
json_parser.get_format_instructions()

'Return a JSON object.'

In [23]:
res  = json_chain.invoke({"question": "What is the biggest country?"})

In [24]:
res

{'answer': 'Russia'}

In [25]:
type(res)

dict

#### Optionally, you can use Pydantic to define a custom output format

In [26]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field

In [27]:
# Define a Pydantic Object with the desired output format.
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")

In [28]:
# Define the parser referring the Pydantic Object
parser = JsonOutputParser(pydantic_object=Joke)

# Add the parser format instructions in the prompt definition.
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# Create a chain with the prompt and the parser
chain = prompt | chatModel | parser

chain.invoke({"query": "Tell me a joke."})

{'setup': 'Why did the scarecrow win an award?',
 'punchline': 'Because he was outstanding in his field!'}